## Uvod 

Ta lekcija bo zajemala: 
- Kaj je klic funkcije in njegove primere uporabe 
- Kako ustvariti klic funkcije z uporabo Azure OpenAI 
- Kako integrirati klic funkcije v aplikacijo 

## Cilji učenja 

Po zaključku te lekcije boste vedeli, kako in razumeli: 

- Namen uporabe klica funkcije 
- Nastavitev klica funkcije z uporabo storitve Azure Open AI 
- Oblikovanje učinkovitih klicev funkcij za primer uporabe vaše aplikacije 


## Razumevanje klicev funkcij 

Za to lekcijo želimo zgraditi funkcijo za naš izobraževalni startup, ki uporabnikom omogoča uporabo klepetalnega robota za iskanje tehničnih tečajev. Priporočali bomo tečaje, ki ustrezajo njihovi ravni znanja, trenutni vlogi in tehnologiji, ki jih zanima. 

Za dokončanje bomo uporabili kombinacijo: 
 - `Azure Open AI` za ustvarjanje klepetalnega doživetja za uporabnika
 - `Microsoft Learn Catalog API` za pomoč uporabnikom pri iskanju tečajev glede na zahtevo uporabnika 
 - `Function Calling` za sprejem poizvedbe uporabnika in pošiljanje te v funkcijo za izvedbo API zahteve. 

Za začetek si poglejmo, zakaj bi želeli sploh uporabljati klice funkcij: 


### Zakaj klic funkcije

Če ste že zaključili katero koli drugo lekcijo v tem tečaju, verjetno razumete moč uporabe velikih jezikovnih modelov (LLM). Upajmo, da lahko vidite tudi nekatere njihove omejitve.

Klic funkcije je funkcija storitve Azure Open AI, ki premaguje naslednje omejitve:
1) Dosleden format odgovora
2) Sposobnost uporabe podatkov iz drugih virov aplikacije v klepetalnem kontekstu

Pred klicem funkcije so bili odgovori LLM neurejeni in nedosledni. Razvijalci so morali napisati zapleteno kodo za preverjanje, da so lahko obdelali vsako različico odgovora.

Uporabniki niso mogli dobiti odgovorov, kot je "Kakšno je trenutno vreme v Stockholmu?". To je zato, ker so bili modeli omejeni na čas, ko so bili podatki izdelani.

Poglejmo si spodnji primer, ki ponazarja ta problem:

Recimo, da želimo ustvariti zbirko podatkov o študentih, da jim lahko predlagamo pravi tečaj. Spodaj imamo dva opisa študentov, ki sta v podatkih zelo podobna.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Želimo to poslati LLM-ju, da obdela podatke. To lahko kasneje uporabimo v naši aplikaciji, da to pošljemo API-ju ali shranimo v podatkovno bazo. 

Ustvarimo dva enaka poziva, v katerih LLM-ju sporočimo, kakšne informacije nas zanimajo: 


Želimo to poslati LLM-ju, da analizira dele, ki so pomembni za naš izdelek. Tako lahko ustvarimo dva identična poziva za usmerjanje LLM-ja: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Po ustvarjanju teh dveh pozivov ju bomo poslali LLM z uporabo `client.responses.create`. Poziv shranimo v spremenljivko `input` in vlogi dodelimo `user`. To je, da posnemamo sporočilo uporabnika, ki ga piše chatbotu. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Zdaj lahko pošljemo oba zahtevka LLM-ju in pregledamo odgovor, ki ga prejmemo. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Čeprav so pozivi enaki in so opisi podobni, lahko dobimo različne formate lastnosti `Grades`. 

Če zgornjo celico zaženete večkrat, je lahko format `3.7` ali `3.7 GPA`. 

To je zato, ker LLM sprejme nestrukturirane podatke v obliki napisanega poziva in vrne prav tako nestrukturirane podatke. Potrebujemo strukturiran format, da vemo, kaj pričakovati pri shranjevanju ali uporabi teh podatkov.

Z uporabo funkcijskega klica lahko zagotovimo, da prejmemo nazaj strukturirane podatke. Pri uporabi funkcijskega klica LLM dejansko ne kliče ali izvaja nobenih funkcij. Namesto tega ustvarimo strukturo, ki jo LLM sledi za svoje odgovore. Nato te strukturirane odgovore uporabimo, da vemo, katero funkcijo naj izvedemo v naših aplikacijah.  
 


![Diagram poteka klica funkcije](../../../../translated_images/sl/Function-Flow.083875364af4f4bb.webp)


Nato lahko vzamemo tisto, kar funkcija vrne, in to pošljemo nazaj LLM. LLM bo nato odgovoril v naravnem jeziku, da bo odgovoril na uporabnikovo vprašanje.


### Primeri uporabe klicev funkcij 

**Klic zunanjih orodij** 
Klepetalni roboti so odlični pri zagotavljanju odgovorov na vprašanja uporabnikov. Z uporabo klicev funkcij lahko klepetalni roboti uporabijo sporočila uporabnikov za dokončanje določenih opravil. Na primer, študent lahko prosi klepetalnega robota, naj "Pošlje e-pošto mojim inštruktorjem, da potrebujem več pomoči pri tem predmetu". To lahko sproži klic funkcije `send_email(to: string, body: string)`


**Ustvarjanje API ali podatkovnih poizvedb**
Uporabniki lahko najdejo informacije z uporabo naravnega jezika, ki se pretvori v formatirano poizvedbo ali API zahtevo. Primer tega je učitelj, ki vpraša "Kateri študenti so opravili zadnjo nalogo", kar lahko kliče funkcijo z imenom `get_completed(student_name: string, assignment: int, current_status: string)`


**Ustvarjanje strukturiranih podatkov**
Uporabniki lahko vzamejo blok besedila ali CSV in uporabijo LLM za izluščitev pomembnih informacij iz njega. Na primer, študent lahko pretvori Wikipedijin članek o mirovnih sporazumih za ustvarjanje AI kartic za učenje. To se lahko naredi z uporabo funkcije, imenovane `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Ustvarjanje vašega prvega klica funkcije 

Postopek ustvarjanja klica funkcije vključuje 3 glavne korake: 
1. Klic API-ja za dokončanja pogovora s seznamom vaših funkcij in sporočilom uporabnika 
2. Preberi odgovor modela za izvedbo dejanja, tj. izvršitev funkcije ali API klica 
3. Naredite še en klic API-ja za dokončanja pogovora z odgovorom vaše funkcije, da uporabite te informacije za ustvarjanje odgovora uporabniku. 


![Tok klica funkcije](../../../../translated_images/sl/LLM-Flow.3285ed8caf4796d7.webp)


### Elementi klica funkcije 

#### Vnos uporabnika 

Prvi korak je ustvariti uporabniško sporočilo. To je mogoče dinamično dodeliti z zajemom vrednosti iz besedilnega vnosa ali pa lahko tukaj dodelite vrednost. Če zaženete API za dokončanje pogovorov prvič, moramo opredeliti `vlogo` in `vsebino` sporočila. 

`Vloga` je lahko `system` (ustvarjanje pravil), `assistant` (model) ali `user` (končni uporabnik). Za klic funkcije bomo to dodelili kot `user` in primer vprašanja. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Ustvarjanje funkcij. 

Naslednje bomo definirali funkcijo in njene parametre. Tukaj bomo uporabili samo eno funkcijo, imenovano `search_courses`, vendar lahko ustvarite več funkcij.

**Pomembno** : Funkcije so vključene v sistemsko sporočilo za LLM in so vključene v količino razpoložljivih žetonov, ki jih imate na voljo. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definicije** 

`name` - Ime funkcije, ki jo želimo poklicati. 

`description` - To je opis, kako funkcija deluje. Pomembno je, da je tukaj specifično in jasno 

`parameters` - Seznam vrednosti in oblik, ki jih želite, da model vključi v svoj odgovor 


`type` - Podatkovni tip, v katerem bodo shranjene lastnosti 

`properties` - Seznam specifičnih vrednosti, ki jih bo model uporabil v svojem odgovoru 


`name` - ime lastnosti, ki jo bo model uporabil v svojem oblikovanem odgovoru 

`type` - Podatkovni tip te lastnosti 

`description` - Opis specifične lastnosti 


**Neobvezno**

`required` - zahtevana lastnost, da je klic funkcije zaključen 


### Klic funkcije
Po definiranju funkcije jo moramo zdaj vključiti v klic API-ja za dopolnjevanje pogovora. To storimo tako, da v zahtevo dodamo `functions`. V tem primeru `functions=functions`.

Obstaja tudi možnost nastavitve `function_call` na `auto`. To pomeni, da bomo prepustili LLM, da odloči, katera funkcija naj bo poklicana glede na uporabnikovo sporočilo, namesto da jo določimo sami.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Zdaj si poglejmo odgovor in kako je oblikovan: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Vidite lahko, da je ime funkcije poklicano in iz uporabnikovega sporočila je LLM uspel najti podatke, ki ustrezajo argumentom funkcije. 


## 3. Integracija funkcijskih klicev v aplikacijo.


Ko smo preizkusili oblikovani odgovor iz LLM, ga lahko sedaj integriramo v aplikacijo.

### Upravljanje s tokom

Za integracijo v našo aplikacijo sledimo naslednjim korakom:

Najprej izvedimo klic na storitve Open AI in sporočilo shranimo v spremenljivko `response_message`.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Zdaj bomo definirali funkcijo, ki bo klicala Microsoft Learn API za pridobitev seznama tečajev: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Kot najboljšo prakso bomo potem preverili, ali model želi poklicati funkcijo. Nato bomo ustvarili eno od razpoložljivih funkcij in jo povezali s funkcijo, ki se kliče. 
Nato bomo vzeli argumente funkcije in jih preslikali na argumente iz LLM.

Nazadnje bomo dodali sporočilo klica funkcije in vrednosti, ki jih je vrnilo sporočilo `search_courses`. To LLM-u daje vse informacije, ki jih potrebuje za
odgovor uporabniku z uporabo naravnega jezika. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Zdaj bomo poslali posodobljeno sporočilo modelu LLM, da bomo prejeli odgovor v naravnem jeziku namesto odgovora v formatu JSON API. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Koda Izziv 

Odlično delo! Za nadaljevanje vašega učenja Azure Open AI Function Calling lahko zgradite: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Več parametrov funkcije, ki lahko pomagajo učenjem najti več tečajev. Razpoložljive parametre API-ja najdete tukaj: 
 - Ustvarite še en klic funkcije, ki od učenca zahteva več informacij, na primer njihov materni jezik 
 - Ustvarite upravljanje napak, ko klic funkcije in/ali klic API-ja ne vrne nobenih ustreznih tečajev 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
